In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score

In [2]:
features_map = {
    "J. Kampe": [
        "z_term_3", "z_term_2", "d_lag_2", "d_lag_3", "d_lag_4", "z_cogram", "d_lag_16", "d_lag_5",
        "d_lag_15", "d_lag_17", "d_lag_12", "d_lag_6", "d_lag_1", "z_term_4", "z_term_6", "d_lag_18",
        "d_lag_13", "d_lag_7", "d_lag_14", "z_cogram_lag_4", "z_gram_lag_4", "z_cogram_lag_5", "z_gram_lag_5", "z_gram_lag_7",
        "z_term_5", "z_gram_lag_6", "z_cogram_lag_6", "z_cogram_lag_7", "d_lag_11", "z_cogram_lag_2", "d_lag_21", "z_cogram_lag_8",
        "z_cogram_lag_11", "z_cogram_lag_3", "d_lag_8", "z_gram_lag_2", "z_gram_lag_8", "z_gram_lag_1", "z_cogram_lag_12", "d_lag_19"
    ]
}

random_forest_features = pd.read_csv("../results/random_forest_feature_selection.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/gevrey_method_feature_selection.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/mrmr_10_features.csv")["feature"].tolist()
mrmr_14_features = pd.read_csv("../results/mrmr_14_features.csv")["feature"].tolist()

features_map["Random Forest"] = random_forest_features
features_map["Correlation"] = correlation_features
features_map["Gevrey Method"] = gevrey_method_features
features_map["Gevrey Method (8 features)"] = gevrey_method_features[:8]    # Limiting to top 8 features
features_map["Gevrey Method (10 features)"] = gevrey_method_features[:10]  # Limiting to top 10 features
features_map["Gevrey Method (12 features)"] = gevrey_method_features[:12]  # Limiting to top 12 features
features_map["Gevrey Method (14 features)"] = gevrey_method_features[:14]  # Limiting to top 14 features
features_map["Gevrey Method (20 features)"] = gevrey_method_features[:20]  # Limiting to top 20 features
features_map["mRMR (10 features)"] = mrmr_10_features
features_map["mRMR (14 features)"] = mrmr_14_features

In [3]:
class DatasetScalerService:
    MAX_LIMIT = 100_000
    def __init__(self, scaler_cls: type[StandardScaler|MinMaxScaler], features: list[str]):
        self.__scaler_X = scaler_cls()
        self.__scaler_y = scaler_cls()
        self.__X_original = pd.read_csv("../dataset/j_kampe.csv")
        self.__y_original = pd.read_csv("../dataset/distances.csv")["distance"]
        self.__features = features

    def get_scaled_data(self, limit: int = 11_000):
        if limit > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.__X_original[self.__features].values
        y = self.__y_original.values.reshape(-1, 1)

        X = X[1_000:limit]
        y = y[1_000:limit]

        X_train = X[:int(0.8 * X.shape[0])]
        X_test  = X[int(0.8 * X.shape[0]):]
        y_train = y[:int(0.8 * y.shape[0])]
        y_test  = y[int(0.8 * y.shape[0]):]

        X_train_scaled = self.__scaler_X.fit_transform(X_train)
        X_test_scaled  = self.__scaler_X.transform(X_test)
        y_train_scaled = self.__scaler_y.fit_transform(y_train).ravel()
        y_test_scaled  = self.__scaler_y.transform(y_test).ravel()
        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled

    @property
    def scaler_y(self) -> StandardScaler|MinMaxScaler:
        return self.__scaler_y

In [ ]:
def train_model(scaler_cls: type[StandardScaler|MinMaxScaler], features_map: dict[str, list[str]]):
    C_values = (0.1, 1.0, 10.0)
    epsilon_values = (0.01, 0.1, 1, 0.02)
    best_results = []

    for name, features in features_map.items():
        best_rmse = np.inf
        best_r2   = -np.inf
        best_result = None
        dataset_scaler_service = DatasetScalerService(scaler_cls, features)
        (
            X_train_scaled,
            X_test_scaled,
            y_train_scaled,
            y_test_scaled
        ) = dataset_scaler_service.get_scaled_data()

        scaler_y = dataset_scaler_service.scaler_y

        for C in C_values:
            for epsilon in epsilon_values:
                model = SVR(kernel="rbf", C=C, epsilon=epsilon)
                model.fit(X_train_scaled, y_train_scaled)
                y_pred_scaled = model.predict(X_test_scaled)

                y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))
                y_test = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1))

                rmse = root_mean_squared_error(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)

                if rmse < best_rmse and r2 > best_r2:
                    best_rmse = rmse
                    best_r2 = r2
                    best_result = {
                        "Group": name,
                        "model": "SVR",
                        "Best RMSE": rmse,
                        "R2": r2,
                        "Best C": C,
                        "Best Epsilon": epsilon,
                        "Features": len(features)
                    }
        if best_result:
            best_results.append(best_result)

    return best_results

# Testando com MinMaxScaler

In [ ]:
best_results = train_model(MinMaxScaler, features_map)
svr_df = pd.DataFrame(best_results).sort_values("Best RMSE").reset_index(drop=True)
print(svr_df)
svr_df.to_csv("../results/svr_experiment_6_minmaxscaler.csv", index=False)

                          Group model  Best RMSE        R2  Best C  \
0   Gevrey Method (20 features)   SVR   0.031655  0.984746    10.0   
1   Gevrey Method (14 features)   SVR   0.032490  0.983931    10.0   
2   Gevrey Method (12 features)   SVR   0.040334  0.975236    10.0   
3                      J. Kampe   SVR   0.061501  0.942423    10.0   
4   Gevrey Method (10 features)   SVR   0.061799  0.941865    10.0   
5                 Random Forest   SVR   0.067228  0.931201    10.0   
6                 Gevrey Method   SVR   0.068517  0.928537    10.0   
7    Gevrey Method (8 features)   SVR   0.072340  0.920341    10.0   
8                   Correlation   SVR   0.091206  0.873371    10.0   
9            mRMR (10 features)   SVR   0.101834  0.842143    10.0   
10           mRMR (14 features)   SVR   0.118905  0.784779     1.0   

    Best Epsilon  Features  
0           0.01        20  
1           0.01        14  
2           0.01        12  
3           0.01        40  
4           0.

# Testando com StandardScaler

In [ ]:
best_results = train_model(StandardScaler, features_map)
svr_df = pd.DataFrame(best_results).sort_values("Best RMSE").reset_index(drop=True)
print(svr_df)
svr_df.to_csv("../results/svr_experiment_6_standardscaler.csv", index=False)

                          Group model  Best RMSE        R2  Best C  \
0   Gevrey Method (20 features)   SVR   0.028549  0.987593    10.0   
1   Gevrey Method (14 features)   SVR   0.031433  0.984959    10.0   
2   Gevrey Method (12 features)   SVR   0.040188  0.975414    10.0   
3                      J. Kampe   SVR   0.042831  0.972074    10.0   
4                 Gevrey Method   SVR   0.044899  0.969313    10.0   
5   Gevrey Method (10 features)   SVR   0.061228  0.942933    10.0   
6                 Random Forest   SVR   0.066653  0.932373    10.0   
7    Gevrey Method (8 features)   SVR   0.071001  0.923261    10.0   
8                   Correlation   SVR   0.085290  0.889266    10.0   
9            mRMR (10 features)   SVR   0.086569  0.885921    10.0   
10           mRMR (14 features)   SVR   0.095567  0.860974    10.0   

    Best Epsilon  Features  
0           0.01        20  
1           0.01        14  
2           0.02        12  
3           0.01        40  
4           0.